# ArcGIS Online Item Terms of Use Editor


**Welcome!**  

This notebook is designed for ArcGIS Online administrators who need to review and edit item Terms of Use at scale. It focuses on the Terms of Use field (`licenseInfo`) and supports a review-first workflow before any edits are applied.

This version bundles `helper_functions.py` and `Esri_ToU.html` template directly into the notebook. During setup, those assets are expanded into `/arcgis/home/notebook_outputs` (or local notebook output paths), where you can inspect inputs and outputs as you work. A review webpage is generated so operators can verify planned changes before execution.

*** BE CAUTIOUS WITH ANY TOOL LIKE THIS THAT BULK EDITS ITEMS *** The workflow intentionally requires explicit review and confirmation before edits and includes an "Undo" process that can be run at a later date if items are found which require reversion. But as usual, YMMV and _caveat emptor_.

**How to use this notebook**  
 - Click on the text "Setup and authenticate" below. 
 - There are two types of cells, Markdown (formatted notes) and Code.
 - An indicator -- typically a vertical blue line -- should highlight that you have selected the "Setup and authenticate" Markdown cell.
 - Once selected, click the "Play" button in the toolbar above to run the cell and advance to the next Code cell.
 - Click the "Play" button a second time to run the code cell.
 - After several seconds a "Setup Notebook" button should appear. Click the button to begin setup and authentication.
 - After each cell completes, click the text within the following Markdown cell.
 - Click the "Play" button to advance to the Code cell, then click the "Play" button a second time to make a button appear.
 - Click the button to run the code in the cell. 

**Notes**  
- Organization-wide scans can take time, especially in large orgs, so progress messages are shown as users are processed.
- Candidate search and structural replacement are separate steps by design:
  - Scan steps find candidate items that contain the terms you specify.
  - Dry-run applies structural matching logic to build the replacement plan.
- You can monitor the status of long running cells by viewing the small circle in the top right of the page.
- If you click on a code cell it will expand showing you the behind-the-scenes Python code.
- For a cleaner interface select View > Collapse All Code in the menu bar above to hide the code .
- If at any point you get stuck and want to start over, just click Kernel > Restart Kernel and Clear Outputs of All Cells... in the menu bar
- The workflow is designed to be safe by default: review first, then edit.


**TL;DR**


In [ ]:
# Run this cell to display Notebook details

from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does (admin workflow)**  
- Authenticates to ArcGIS Online.
- Scans an entire organization's item Terms of Use (`licenseInfo`) for operator-defined terms.
- Identifies candidate items first, then separately builds a structural dry-run replacement plan.
- Exports scan outputs for auditability to a file (default filename: `scan_results.csv`).
- Generates a side-by-side HTML review report for operator QA.
- Applies edits only after explicit confirmation.
- Provides an optional UNDO operation to revert changes if needed.
- Exports edit results for record-keeping.
- Tested in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))


## 1. Setup and authenticate
Connect to ArcGIS Online and initialize the notebook environment.


In [ ]:
# Bootstrap bundled assets for the portable notebook.
import base64
import sys
from pathlib import Path

OUTPUT_DIR_NAME = "notebook_outputs"
RUNTIME_ROOT = Path("/arcgis/home") if Path("/arcgis/home").exists() else Path.cwd()
RUNTIME_DIR = (RUNTIME_ROOT / OUTPUT_DIR_NAME).resolve()
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
HELPER_FUNCTIONS_B64 = (
    "..."
)
ESRI_TOU_HTML_B64 = (
    "PHA+CiAgICA8aW1nIHNyYz0iaHR0cHM6Ly93d3cuZXNyaS5jb20vY29udGVudC9kYW0vZXNyaXNpdGVzL2VuLXVzL2NvbW1vbi9sb2dvcy9lc3JpLWxvZ28u"
    "anBnIiBhbHQ9IkVzcmkgbG9nbyIgd2lkdGg9IjExMyIgaGVpZ2h0PSIzOSI+CjwvcD4KPHAgc3R5bGU9ImNvbG9yOnJnYig3NCw3NCw3NCk7Zm9udC1mYW1p"
    "bHk6J0F2ZW5pciBOZXh0JyxBdmVuaXIsJ0hlbHZldGljYSBOZXVlJyxzYW5zLXNlcmlmO2ZvbnQtc2l6ZToxNnB4O21hcmdpbjowIDAgMXJlbTsiPgogICAg"
    "VGhpcyB3b3JrIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBFc3JpIE1hc3RlciBMaWNlbnNlIEFncmVlbWVudC4KPC9wPgo8cCBzdHlsZT0iY29sb3I6cmdiKDc0"
    "LDc0LDc0KTtmb250LWZhbWlseTonQXZlbmlyIE5leHQnLEF2ZW5pciwnSGVsdmV0aWNhIE5ldWUnLHNhbnMtc2VyaWY7Zm9udC1zaXplOjE2cHg7bWFyZ2lu"
    "OjA7Ij4KICAgIDxhIHN0eWxlPSJjb2xvcjpyZ2IoMCw5NywxNTUpO3RleHQtZGVjb3JhdGlvbjpub25lOyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVu"
    "ZXIgbm9yZWZlcnJlciIgaHJlZj0iaHR0cHM6Ly9nb3RvLmFyY2dpcy5jb20vdGVybXNvZnVzZS92aWV3c3VtbWFyeSI+PHN0cm9uZz5WaWV3IFN1bW1hcnk8"
    "L3N0cm9uZz48L2E+IHwgPGEgc3R5bGU9ImNvbG9yOnJnYigwLDk3LDE1NSk7dGV4dC1kZWNvcmF0aW9uOm5vbmU7IiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJu"
    "b29wZW5lciBub3JlZmVycmVyIiBocmVmPSJodHRwczovL2dvdG8uYXJjZ2lzLmNvbS90ZXJtc29mdXNlL3ZpZXd0ZXJtc29mdXNlIj48c3Ryb25nPlZpZXcg"
    "VGVybXMgb2YgVXNlPC9zdHJvbmc+PC9hPgo8L3A+"
)

BOOTSTRAP_FILES = {
    "helper_functions.py": base64.b64decode(HELPER_FUNCTIONS_B64).decode("utf-8"),
    "Esri_ToU.html": base64.b64decode(ESRI_TOU_HTML_B64).decode("utf-8"),
}

for filename, file_text in BOOTSTRAP_FILES.items():
    target_path = RUNTIME_DIR / filename
    target_path.write_text(file_text, encoding="utf-8")
    print(f"Bootstrapped asset: {target_path}")

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print(f"Portable notebook assets are ready in: {RUNTIME_DIR}")

# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.

print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

NOTEBOOK_DIR = Path.cwd().resolve()
IS_PORTABLE_NOTEBOOK = "RUNTIME_DIR" in globals()

if IS_PORTABLE_NOTEBOOK:
    # Portable notebook: prefer freshly bootstrapped assets in notebook_outputs.
    candidate_helper_dirs = [
        Path(globals()["RUNTIME_DIR"]).resolve(),
        NOTEBOOK_DIR / "notebook_outputs",
        NOTEBOOK_DIR,
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]
else:
    # Source notebook: prefer repository files first to avoid stale bootstrapped copies.
    candidate_helper_dirs = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR / ".local_testing" / "notebook_outputs",
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]

def _find_helper_dir(candidate_dirs):
    for p in candidate_dirs:
        helper_file = p / "helper_functions.py"
        if helper_file.exists():
            return p.resolve()
    return None

selected_helper_dir = _find_helper_dir(candidate_helper_dirs)

if selected_helper_dir is None and IS_PORTABLE_NOTEBOOK and "BOOTSTRAP_FILES" in globals():
    runtime_dir = Path(globals().get("RUNTIME_DIR", NOTEBOOK_DIR / "notebook_outputs")).resolve()
    runtime_dir.mkdir(parents=True, exist_ok=True)
    for filename, file_text in globals()["BOOTSTRAP_FILES"].items():
        target_path = runtime_dir / filename
        target_path.write_text(file_text, encoding="utf-8")
        print(f"Recreated missing bundled asset: {target_path}")
    selected_helper_dir = _find_helper_dir(candidate_helper_dirs)

if selected_helper_dir is None:
    raise FileNotFoundError(
        "Could not locate helper_functions.py in expected locations. "
        "For source notebook runs, keep helper_functions.py in the notebook folder. "
        "For portable runs, execute the bootstrap section first."
    )

# Ensure the selected helper path wins over stale entries.
selected_helper_dir_str = str(selected_helper_dir)
sys.path[:] = [x for x in sys.path if x != selected_helper_dir_str]
sys.path.insert(0, selected_helper_dir_str)

# Force reload source if helper_functions was previously imported from another location.
if "helper_functions" in sys.modules:
    del sys.modules["helper_functions"]

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    bind_button_with_status,
    bind_primary_scan_with_cancel,
    setup_notebook_btn,
    save_scan_outputs_btn,
    load_previous_scan_btn,
    run_dry_run_with_file_btn,
    preview_dry_run_match_btn,
    create_report_btn,
    export_dry_run_btn,
    load_update_selection_btn,
    apply_updates_btn,
    export_final_results_btn,
    refresh_scan_save_ui,
    refresh_dry_run_export_ui,
    load_rollback_snapshot_btn,
    preview_rollback_btn,
    execute_rollback_btn,
    refresh_rollback_export_ui,
    refresh_rollback_target_mode_ui,
    export_rollback_results_btn,
    OFFICIAL_TOU_HTML_FILE,
    )

# Resolve the canonical ToU template path based on notebook mode.
if IS_PORTABLE_NOTEBOOK:
    resolved_tou_path = (selected_helper_dir / "Esri_ToU.html").resolve()
else:
    resolved_tou_path = (NOTEBOOK_DIR / "Esri_ToU.html").resolve()
if not resolved_tou_path.exists():
    resolved_tou_path = Path(OFFICIAL_TOU_HTML_FILE).resolve()

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": str(resolved_tou_path),
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

if not IS_PORTABLE_NOTEBOOK:
    print(f"Helper path: {selected_helper_dir}")
    print(f"Official ToU template file: {context['official_tou_html_file']}")

setup_output = initialize_ui("output")
context["setup_output"] = setup_output
auth_container = widgets.VBox([])
context["auth_container"] = auth_container

# Create widgets
setup_button = initialize_ui("button", description="Setup Notebook", width="200px")
setup_status = widgets.HTML(value="")
context["setup_status"] = setup_status
display(widgets.HBox([setup_button, setup_status]))
bind_button_with_status(
    setup_button,
    setup_notebook_btn,
    "setup_status",
    "Setup in progress... please wait.",
    "Setup complete.",
    "Setup failed. See details below.",
    output_key="setup_output",
)
display(setup_output)
display(auth_container)


## 2. Scan your content
Search your organization for candidate items containing the text strings and/or HTML entered below.
This step is candidate discovery only; structural replacement matching happens later in the dry-run step.
There is an optional cap: leave it blank to scan the entire org, or enter a number to stop after that many matches for faster test runs.
After the scan finishes, optional save fields appear below when there is scan output worth exporting.


In [ ]:
# Cell 2: Scan org content for licenseInfo matches and optionally save the results

scan_output = initialize_ui("output")
context["scan_output"] = scan_output

scan_help_html = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt; link &lt;/a&gt;&quot;.<br>"
        "Formatting will automatically be bundled for processing when you click the button."
        "</div>"
    )
)

scan_terms_input = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["scan_terms_input"] = scan_terms_input
scan_limit_input = initialize_ui(
    "text",
    value="",
    description="Stop scan after X matches (optional):",
    width="300px",
)
context["scan_limit_input"] = scan_limit_input
scan_button = initialize_ui("button", description="Scan for items", width="200px")
scan_status = widgets.HTML(value="")
context["scan_status"] = scan_status

save_scan_output = initialize_ui("output")
context["save_scan_output"] = save_scan_output
scan_results_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_results.csv"),
    description="Output scan CSV:",
    width="700px",
)
context["scan_results_path_input"] = scan_results_path_input
save_scan_button = initialize_ui("button", description="Save file")
context["save_scan_button"] = save_scan_button
save_scan_status = widgets.HTML(value="")
context["save_scan_status"] = save_scan_status
save_scan_container = widgets.VBox([])
context["save_scan_container"] = save_scan_container

context["refresh_scan_save_ui"] = refresh_scan_save_ui
refresh_scan_save_ui()

display(
    widgets.VBox([
        scan_help_html,
        scan_terms_input,
        scan_limit_input,
        widgets.HBox([scan_button, scan_status]),
        scan_output,
        save_scan_container,
    ])
)

bind_primary_scan_with_cancel(
    scan_button,
    status_key="scan_status",
    output_key="scan_output",
    input_key="scan_terms_input",
    limit_input_key="scan_limit_input",
)

bind_button_with_status(
    save_scan_button,
    save_scan_outputs_btn,
    "save_scan_status",
    "Save in progress... please wait.",
    "Save complete.",
    "Save failed. See details below.",
    output_key="save_scan_output",
)


## 3. Reload results from a previous scan (optional)
Load previously saved CSV files so you can continue working without running a new scan.


In [ ]:
# Cell 3: Optionally load results from a previous scan

reload_scan_output = initialize_ui("output")
context["reload_scan_output"] = reload_scan_output

reload_scan_results_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_results.csv"),
    description="Input scan CSV:",
    width="900px",
)
context["reload_scan_results_path_input"] = reload_scan_results_path_input

reload_scan_button = initialize_ui("button", description="Load previous scan file", width="240px")
reload_scan_status = widgets.HTML(value="")
context["reload_scan_status"] = reload_scan_status

display(
    widgets.VBox([
        reload_scan_results_path_input,
        widgets.HBox([reload_scan_button, reload_scan_status]),
        reload_scan_output,
    ])
)

bind_button_with_status(
    reload_scan_button,
    load_previous_scan_btn,
    "reload_scan_status",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="reload_scan_output",
)


## 4. Dry run
Do a dry-run before making any changes. Modify the input file to use your own custom HTML that will replace the search terms.
By default, the edit logic uses a semi-greedy matcher: it looks for a recognized Esri logo, then scans forward within bounded distance for the license text and the related links. After a replacement is made, cleanup includes removing trivial wrapper junk around the replacement block.

If you enable strict matching below, the dry run will only match blocks that contain your search text exactly. Use strict mode when you want a more conservative replacement plan.

After the dry run finishes, an optional CSV export appears when there is output worth saving.


In [ ]:
# Cell 4: Do a dry-run before making any changes and optionally export the plan

dry_run_output = initialize_ui("output")
context["dry_run_output"] = dry_run_output
dry_run_preview_output = initialize_ui("output")
context["dry_run_preview_output"] = dry_run_preview_output
dry_run_template_path_input = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
context["dry_run_template_path_input"] = dry_run_template_path_input
strict_match_checkbox = initialize_ui(
    "checkbox",
    description="Enforce strict matching?",
    value=False,
    width="320px",
)
context["strict_match_checkbox"] = strict_match_checkbox
dry_run_preview_button = initialize_ui("button", description="Preview card comparison", width="220px")
dry_run_preview_status = widgets.HTML(value="")
context["dry_run_preview_status"] = dry_run_preview_status
dry_run_button = initialize_ui("button", description="Do a dry run", width="180px")
dry_run_status = widgets.HTML(value="")
context["dry_run_status"] = dry_run_status

dry_run_export_output = initialize_ui("output")
context["dry_run_export_output"] = dry_run_export_output
dry_run_export_path_input = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["dry_run_export_path_input"] = dry_run_export_path_input
dry_run_export_button = initialize_ui("button", description="Export dry-run CSV", width="200px")
context["dry_run_export_button"] = dry_run_export_button
dry_run_export_status = widgets.HTML(value="")
context["dry_run_export_status"] = dry_run_export_status
dry_run_export_container = widgets.VBox([])
context["dry_run_export_container"] = dry_run_export_container

context["refresh_dry_run_export_ui"] = refresh_dry_run_export_ui
refresh_dry_run_export_ui()

display(
    widgets.VBox([
        dry_run_template_path_input,
        strict_match_checkbox,
        widgets.HBox([dry_run_preview_button, dry_run_preview_status]),
        dry_run_preview_output,
        widgets.HBox([dry_run_button, dry_run_status]),
        dry_run_output,
        dry_run_export_container,
    ])
)

bind_button_with_status(
    dry_run_preview_button,
    preview_dry_run_match_btn,
    "dry_run_preview_status",
    "Preview generation in progress... please wait.",
    "Preview ready.",
    "Preview failed. See details below.",
    output_key="dry_run_preview_output",
)

bind_button_with_status(
    dry_run_button,
    run_dry_run_with_file_btn,
    "dry_run_status",
    "Dry run in progress... please wait.",
    "Dry run complete.",
    "Dry run failed. See details below.",
    output_key="dry_run_output",
)

bind_button_with_status(
    dry_run_export_button,
    export_dry_run_btn,
    "dry_run_export_status",
    "Dry-run export in progress... please wait.",
    "Dry-run export complete.",
    "Dry-run export failed. See details below.",
    output_key="dry_run_export_output",
)


## 5. Create a report to review before applying edits
A HTML file will be created. Large reports cannot be properly displayed in the output cell. When this happens, download the file from /arcgis/home/notebook_outputs by clicking on the filename. Then open that file in your local browser. You'll then make selections, save a CSV file and upload that file to /arcgis/home/notebook_outputs for the final edit step.
There is an optional cap: leave it blank to include all planned edits, or enter a number to generate a smaller review report for faster testing.


In [ ]:
# Cell 5: Generate an HTML report for review before applying edits

create_report_output = initialize_ui("output")
context["create_report_output"] = create_report_output
report_path_input = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["report_path_input"] = report_path_input
selection_ids_filename_input = initialize_ui(
    "text",
    value=Path(default_output_path_str("selected_item_ids.csv")).name,
    description="Base filename generated when downloading IDs after review:",
    width="700px",
)
context["selection_ids_filename_input"] = selection_ids_filename_input
create_report_button = initialize_ui("button", description="Create report", width="200px")
create_report_status = widgets.HTML(value="")
context["create_report_status"] = create_report_status

display(
    widgets.VBox([
        report_path_input,
        selection_ids_filename_input,
        widgets.HBox([create_report_button, create_report_status]),
        create_report_output,
    ])
)

bind_button_with_status(
    create_report_button,
    create_report_btn,
    "create_report_status",
    "Report generation in progress... please wait.",
    "Report generation complete.",
    "Report generation failed. See details below.",
    output_key="create_report_output",
)


## 6. Apply edits
Use this step to safely confirm what will be edited before making any changes.
1. Enter the CSV file path that contains item IDs selected from the report.
2. (Optional but recommended) Set the undo snapshot CSV output path and filename for this edit run.
3. Click **Load file** to preview how many items will be edited.
4. Review the summary shown in the output area.
5. If it looks correct, type `APPLY EDITS` in the confirmation box.
6. Click **Execute edit** to apply the edits.


In [ ]:
# Cell 6: Apply edits only after reviewing the dry run report 

apply_edits_output = initialize_ui("output")
context["apply_edits_output"] = apply_edits_output
selected_ids_to_edit_path_input = initialize_ui(
    "text",
    value=Path(default_output_path_str("selected_item_ids.csv")).name,
    description="CSV file with item IDs to edit:",
    width="700px",
)
context["selected_ids_to_edit_path_input"] = selected_ids_to_edit_path_input

undo_snapshot_path_input = initialize_ui(
    "text",
    value=default_output_path_str("rollback_snapshot.csv"),
    description="Undo snapshot CSV output:",
    width="700px",
)
context["undo_snapshot_path_input"] = undo_snapshot_path_input

load_selected_ids_button = initialize_ui("button", description="Load file", width="140px")
load_selected_ids_status = widgets.HTML(value="")
context["load_selected_ids_status"] = load_selected_ids_status

apply_edits_confirmation_input = initialize_ui(
    "text",
    value="",
    description="Type APPLY EDITS to confirm:",
    width="700px",
)
context["apply_edits_confirmation_input"] = apply_edits_confirmation_input

apply_edits_button = initialize_ui("button", description="Execute edit", width="180px")
apply_edits_status = widgets.HTML(value="")
context["apply_edits_status"] = apply_edits_status
display(
    widgets.VBox([
        selected_ids_to_edit_path_input,
        undo_snapshot_path_input,
        widgets.HBox([load_selected_ids_button, load_selected_ids_status]),
        apply_edits_output,
        apply_edits_confirmation_input,
        widgets.HBox([apply_edits_button, apply_edits_status]),
    ])
)

bind_button_with_status(
    load_selected_ids_button,
    load_update_selection_btn,
    "load_selected_ids_status",
    "Loading file and previewing selection... please wait.",
    "File loaded. Review summary below.",
    "Load failed. See details below.",
    output_key="apply_edits_output",
)

bind_button_with_status(
    apply_edits_button,
    apply_updates_btn,
    "apply_edits_status",
    "Edit in progress... please wait.",
    "Edit complete.",
    "Edit failed. See details below.",
    output_key="apply_edits_output",
)


## 7. Undo edits (optional)
Undo selected edits using snapshot rows captured during Step 6.
Use the controls below in this order:
1. Load an undo snapshot CSV from a prior edit run. The file must contain at least `item_id` and `pre_edit_licenseInfo` columns.
2. Choose one undo target mode:
   - `Manual item IDs`: paste item IDs into the text box using comma, space, or newline separators.
   - `IDs file (.csv)`: provide a CSV file of item IDs to target.
3. File schema requirements:
   - CSV must contain an `item_id` column.
4. Click **Preview card comparison** to confirm the exact undo scope and visual before/after preview.
5. Type `APPLY UNDO` and click **Execute undo** only after reviewing the preview.


In [ ]:
# Cell 7: Optional targeted rollback with preview, confirmation, and export

rollback_output = initialize_ui("output")
context["rollback_output"] = rollback_output

rollback_snapshot_path_input = initialize_ui(
    "text",
    value=context.get("rollback_snapshot_path", default_output_path_str("rollback_snapshot.csv")),
    description="Undo snapshot CSV:",
    width="700px",
)
context["rollback_snapshot_path_input"] = rollback_snapshot_path_input
load_rollback_snapshot_button = initialize_ui("button", description="Load snapshot", width="160px")
load_rollback_snapshot_status = widgets.HTML(value="")
context["load_rollback_snapshot_status"] = load_rollback_snapshot_status
load_snapshot_row = widgets.HBox([load_rollback_snapshot_button, load_rollback_snapshot_status])

undo_mode_separator = widgets.HTML(value="<hr style='margin:8px 0 12px 0; border:none; border-top:1px solid #d0d7de;' />")

rollback_mode_help = widgets.HTML(
    value=(
        "<div style='margin:0 0 8px 0; line-height:1.35;'>"
        "Choose one undo data source: Manual item IDs or file. Inactive inputs will be disabled automatically."
        "</div>"
    )
)

rollback_target_mode = initialize_ui(
    "dropdown",
    value="manual",
    description="Undo target mode:",
    options=[
        ("Manual item IDs", "manual"),
        ("IDs file (.csv)", "file"),
    ],
    width="420px",
)
context["rollback_target_mode"] = rollback_target_mode

rollback_ids_text_input = initialize_ui(
    "textarea",
    value="",
    description="Manual item IDs:",
    width="700px",
    height="80px",
)
context["rollback_ids_text_input"] = rollback_ids_text_input
rollback_ids_file_path_input = initialize_ui(
    "text",
    value=Path(default_output_path_str("selected_item_ids.csv")).name,
    description="Target IDs file (.csv):",
    width="700px",
)
context["rollback_ids_file_path_input"] = rollback_ids_file_path_input

preview_rollback_button = initialize_ui("button", description="Preview card comparison", width="220px")
preview_rollback_status = widgets.HTML(value="")
context["preview_rollback_status"] = preview_rollback_status

rollback_confirmation_input = initialize_ui(
    "text",
    value="",
    description="Type APPLY UNDO to confirm:",
    width="700px",
)
context["rollback_confirmation_input"] = rollback_confirmation_input
execute_rollback_button = initialize_ui("button", description="Execute undo", width="180px")
execute_rollback_status = widgets.HTML(value="")
context["execute_rollback_status"] = execute_rollback_status

rollback_export_output = initialize_ui("output")
context["rollback_export_output"] = rollback_export_output
rollback_results_path_input = initialize_ui(
    "text",
    value=default_output_path_str("rollback_results.csv"),
    description="Undo results CSV:",
    width="700px",
)
context["rollback_results_path_input"] = rollback_results_path_input
rollback_export_button = initialize_ui("button", description="Export undo CSV", width="180px")
context["rollback_export_button"] = rollback_export_button
rollback_export_status = widgets.HTML(value="")
context["rollback_export_status"] = rollback_export_status
rollback_export_container = widgets.VBox([])
context["rollback_export_container"] = rollback_export_container

def refresh_rollback_snapshot_load_ui():
    """Hide Load snapshot when Step 7 path matches the latest Step 6-generated snapshot path."""
    saved_snapshot_path = str(context.get("rollback_snapshot_path") or "").strip()
    entered_snapshot_path = str(rollback_snapshot_path_input.value or "").strip()

    saved_norm = str(Path(saved_snapshot_path).expanduser().resolve()) if saved_snapshot_path else ""
    entered_norm = str(Path(entered_snapshot_path).expanduser().resolve()) if entered_snapshot_path else ""
    hide_load = bool(saved_norm and entered_norm and saved_norm == entered_norm)
    load_snapshot_row.layout.display = "none" if hide_load else ""

context["refresh_rollback_export_ui"] = refresh_rollback_export_ui
context["refresh_rollback_target_mode_ui"] = refresh_rollback_target_mode_ui
context["refresh_rollback_snapshot_load_ui"] = refresh_rollback_snapshot_load_ui
refresh_rollback_target_mode_ui()
refresh_rollback_export_ui()
refresh_rollback_snapshot_load_ui()
rollback_target_mode.observe(refresh_rollback_target_mode_ui, names="value")
rollback_snapshot_path_input.observe(lambda *_: refresh_rollback_snapshot_load_ui(), names="value")

display(
    widgets.VBox([
        widgets.HBox([rollback_snapshot_path_input]),
        load_snapshot_row,
        undo_mode_separator,
        rollback_mode_help,
        rollback_target_mode,
        rollback_ids_text_input,
        rollback_ids_file_path_input,
        widgets.HBox([preview_rollback_button, preview_rollback_status]),
        rollback_output,
        rollback_export_container,
        rollback_confirmation_input,
        widgets.HBox([execute_rollback_button, execute_rollback_status]),
    ])
)

bind_button_with_status(
    load_rollback_snapshot_button,
    load_rollback_snapshot_btn,
    "load_rollback_snapshot_status",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="rollback_output",
)

bind_button_with_status(
    preview_rollback_button,
    preview_rollback_btn,
    "preview_rollback_status",
    "Preview generation in progress... please wait.",
    "Preview ready.",
    "Preview failed. See details below.",
    output_key="rollback_output",
)

bind_button_with_status(
    execute_rollback_button,
    execute_rollback_btn,
    "execute_rollback_status",
    "Undo in progress... please wait.",
    "Undo complete.",
    "Undo failed. See details below.",
    output_key="rollback_output",
)

bind_button_with_status(
    rollback_export_button,
    export_rollback_results_btn,
    "rollback_export_status",
    "Undo export in progress... please wait.",
    "Undo export complete.",
    "Undo export failed. See details below.",
    output_key="rollback_export_output",
)


## 8. Export results of the editing process
Export CSV files for record-keeping.
If you ran optional undo in Step 7, use its export control there to generate undo audit results.


In [ ]:
# Cell 8: Export final edit results to a single CSV file for record-keeping

export_final_results_output = initialize_ui("output")
context["export_final_results_output"] = export_final_results_output
final_results_path_input = initialize_ui(
    "text",
    value=default_output_path_str("edit_results.csv"),
    description="Output final edit results CSV:",
    width="700px",
)
context["final_results_path_input"] = final_results_path_input
export_final_results_button = initialize_ui("button", description="Export final CSV", width="180px")
export_final_results_status = widgets.HTML(value="")
context["export_final_results_status"] = export_final_results_status

success_df = context.get("success_df")
update_errors_df = context.get("update_errors_df")

final_export_children = []
if (
    success_df is not None
    and update_errors_df is not None
    and (not success_df.empty or not update_errors_df.empty)
):
    final_export_children.append(final_results_path_input)
    final_export_children.append(widgets.HBox([export_final_results_button, export_final_results_status]))
else:
    final_export_children.append(
        widgets.HTML(
            value="<div style='margin:0; padding:0;'>No non-empty final edit result files are available to export yet.</div>"
        )
    )

final_export_children.append(export_final_results_output)
display(widgets.VBox(final_export_children))

bind_button_with_status(
    export_final_results_button,
    export_final_results_btn,
    "export_final_results_status",
    "Final export in progress... please wait.",
    "Final export complete.",
    "Final export failed. See details below.",
    output_key="export_final_results_output",
)
